Ячейка 1 — импорты и базовые настройки

In [43]:
# Импорты для анализа данных, SQL и визуализаций
import sqlite3
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# Настройки отображения таблиц
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


Ячейка 2 — подключение к SQLite и быстрая проверка таблиц

In [45]:
# Подключаемся к базе SQLite (из notebooks путь такой)
DB_PATH = "../data/ga.sqlite"
con = sqlite3.connect(DB_PATH)

# Проверяем какие таблицы и вьюхи есть в базе
pd.read_sql("""
SELECT type, name 
FROM sqlite_master 
WHERE type IN ('table','view') 
ORDER BY type, name;
""", con)


,type,name
0,table,hits
1,table,session_features
2,table,sessions
3,table,sqlite_sequence


Ячейка 3 — определяем целевые event_action (из методички)

In [46]:
# Список целевых событий (event_action) из методички
TARGET_ACTIONS = [
    "sub_car_claim_click",
    "sub_car_claim_submit_click",
    "sub_open_dialog_click",
    "sub_custom_question_submit_click",
    "sub_call_number_click",
    "sub_callback_submit_click",
    "sub_submit_success",
    "sub_car_request_submit_click",
]

TARGET_ACTIONS


['sub_car_claim_click',
 'sub_car_claim_submit_click',
 'sub_open_dialog_click',
 'sub_custom_question_submit_click',
 'sub_call_number_click',
 'sub_callback_submit_click',
 'sub_submit_success',
 'sub_car_request_submit_click']

Ячейка 4 — создаём “витрину” для DA: target по сессии + атрибуты визита

In [47]:
# ВАЖНО:
# Мы делаем витрину прямо из SQLite (у нас уже есть session_features),
# но для DA нам нужно гарантированно собрать target по hits.

# 1) target по сессии: был ли хотя бы один целевой event_action
target_sql = f"""
WITH target_sessions AS (
    SELECT
        session_id,
        MAX(CASE WHEN event_action IN ({",".join([f"'{a}'" for a in TARGET_ACTIONS])}) THEN 1 ELSE 0 END) AS target
    FROM hits
    WHERE hit_type = 'event'
    GROUP BY session_id
)
SELECT
    sf.*,
    COALESCE(ts.target, 0) AS target
FROM session_features sf
LEFT JOIN target_sessions ts USING(session_id);
"""

df = pd.read_sql(target_sql, con)

# Проверяем размер и первые строки
print("DA dataset shape:", df.shape)
df.head()


DA dataset shape: (1860042, 17)


,session_id,visit_date,utm_source,utm_medium,utm_campaign,device_category,device_os,geo_country,geo_city,hits_cnt,uniq_actions_cnt,has_go_to_car_card,has_showed_number_ads,has_quiz_show,target_showed_number_ads,target_funnel_go_to_then_showed,target
0,9055434745589932991.1637753792.1637753792,2021-11-24,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,mobile,Android,Russia,Zlatoust,2,2,0,0,0,0,0,0
1,905544597018549464.1636867290.1636867290,2021-11-14,MvfHsxITijuriZxsqZqt,cpm,FTjNLDyTrXaWYgZymFkV,mobile,Android,Russia,Moscow,1,1,0,0,0,0,0,0
2,9055446045651783499.1640648526.1640648526,2021-12-28,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,mobile,Android,Russia,Krasnoyarsk,16,3,0,0,0,0,0,0
3,9055447046360770272.1622255328.1622255328,2021-05-29,kjsLglQLzykiRbcDiGcD,cpc,None,mobile,None,Russia,Moscow,3,2,0,0,0,0,0,0
4,9055447046360770272.1622255345.1622255345,2021-05-29,kjsLglQLzykiRbcDiGcD,cpc,None,mobile,None,Russia,Moscow,2,2,0,0,1,0,0,0


Ячейка 5 — базовые sanity-checks (пропуски, типы, base rate)

In [48]:
# Проверяем долю целевого действия (base rate) и пропуски
base_rate = df["target"].mean()
print("Base rate (CR по target):", base_rate)

# Пропуски по колонкам (топ-20)
(df.isna().mean().sort_values(ascending=False).head(20))


Base rate (CR по target): 0.027049926829609224


device_os                          0.575330
utm_campaign                       0.118063
utm_source                         0.000052
session_id                         0.000000
uniq_actions_cnt                   0.000000
target_funnel_go_to_then_showed    0.000000
target_showed_number_ads           0.000000
has_quiz_show                      0.000000
has_showed_number_ads              0.000000
has_go_to_car_card                 0.000000
geo_city                           0.000000
hits_cnt                           0.000000
visit_date                         0.000000
geo_country                        0.000000
device_category                    0.000000
utm_medium                         0.000000
target                             0.000000
dtype: float64

✅ Часть 1. Проверка гипотез (CR сравнения)
Гипотеза A: органика vs платный — нет различий в CR
Ячейка 6 — создаём флаг organic/paid

In [49]:
# Делаем признак "канал": органика vs платный
# Логика:
# - organic: utm_medium is NULL OR utm_medium in ('organic', '(none)') — чаще всего так
# - paid: всё остальное
# Если у тебя в данных другие значения — мы их посмотрим в отдельной ячейке ниже.

df["utm_medium_filled"] = df["utm_medium"].fillna("(none)").str.lower()

df["traffic_type"] = np.where(
    df["utm_medium_filled"].isin(["organic", "(none)"]),
    "organic",
    "paid"
)

df["traffic_type"].value_counts(dropna=False)


traffic_type
paid       1496433
organic     363609
Name: count, dtype: int64

Ячейка 7 — сравнение CR и z-test для долей

In [50]:
# Сравним CR (target mean) между organic и paid
cr_by_traffic = df.groupby("traffic_type")["target"].agg(["count", "mean"])
cr_by_traffic.rename(columns={"mean": "cr"}, inplace=True)
cr_by_traffic


,count,cr
traffic_type,,
organic,363609,0.034738
paid,1496433,0.025182


In [51]:
# Стат-тест для разницы долей: z-test (приближение нормальным распределением)
def two_proportions_ztest(success_a, n_a, success_b, n_b):
    p1 = success_a / n_a
    p2 = success_b / n_b
    p_pool = (success_a + success_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
    z = (p1 - p2) / se
    # двусторонний p-value
    p_value = 2 * (1 - 0.5 * (1 + np.math.erf(abs(z)/np.sqrt(2))))
    return z, p_value, p1, p2

# Достаём числа
a = df[df["traffic_type"] == "organic"]["target"]
b = df[df["traffic_type"] == "paid"]["target"]

z, pval, p1, p2 = two_proportions_ztest(a.sum(), len(a), b.sum(), len(b))
print(f"Organic CR={p1:.5f}, Paid CR={p2:.5f}")
print(f"Z={z:.3f}, p-value={pval:.5g}")


Organic CR=0.03474, Paid CR=0.02518
Z=31.859, p-value=0


/var/folders/0q/s19p92dx2lx90d05rffjvmyw0000gn/T/ipykernel_15876/2329886848.py:9: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (Deprecated Numpy 1.25). Replace usages of `np.math` with `math`
  p_value = 2 * (1 - 0.5 * (1 + np.math.erf(abs(z)/np.sqrt(2))))


Гипотеза B: mobile vs desktop — нет различий в CR
Ячейка 8 — сравнение CR по device_category + тест

In [52]:
# Смотрим распределение устройств
df["device_category"].value_counts(dropna=False)


device_category
mobile     1474871
desktop     366863
tablet       18308
Name: count, dtype: int64

In [53]:
# Оставим только mobile и desktop (если есть tablet, его можно отдельно или убрать)
df_dev = df[df["device_category"].isin(["mobile", "desktop"])].copy()

cr_by_device = df_dev.groupby("device_category")["target"].agg(["count", "mean"])
cr_by_device.rename(columns={"mean": "cr"}, inplace=True)
cr_by_device


,count,cr
device_category,,
desktop,366863,0.031382
mobile,1474871,0.026022


In [54]:
# z-test: mobile vs desktop
m = df_dev[df_dev["device_category"] == "mobile"]["target"]
d = df_dev[df_dev["device_category"] == "desktop"]["target"]

z, pval, p1, p2 = two_proportions_ztest(m.sum(), len(m), d.sum(), len(d))
print(f"Mobile CR={p1:.5f}, Desktop CR={p2:.5f}")
print(f"Z={z:.3f}, p-value={pval:.5g}")


Mobile CR=0.02602, Desktop CR=0.03138
Z=-17.897, p-value=0


/var/folders/0q/s19p92dx2lx90d05rffjvmyw0000gn/T/ipykernel_15876/2329886848.py:9: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (Deprecated Numpy 1.25). Replace usages of `np.math` with `math`
  p_value = 2 * (1 - 0.5 * (1 + np.math.erf(abs(z)/np.sqrt(2))))


Гипотеза C: Москва+область и СПб vs другие — нет различий в CR
Ячейка 9 — создаём флаг “город присутствия” и сравнение

In [55]:
# Флаг: города присутствия (Москва/МО + Санкт-Петербург)
# Уточнение: в данных город может быть записан по-разному.
# Поэтому делаем "мягкое" сравнение через contains.

geo = df["geo_city"].fillna("").str.lower()

df["is_presence_city"] = np.where(
    geo.str.contains("moscow") | geo.str.contains("москва") |
    geo.str.contains("saint petersburg") | geo.str.contains("санкт-петербург") |
    geo.str.contains("st. petersburg") | geo.str.contains("petersburg"),
    1, 0
)

df["is_presence_city"].value_counts()


is_presence_city
1    1102118
0     757924
Name: count, dtype: int64

In [56]:
# CR по presence / non-presence
cr_by_geo = df.groupby("is_presence_city")["target"].agg(["count", "mean"])
cr_by_geo.rename(columns={"mean": "cr"}, inplace=True)
cr_by_geo.index = ["other_regions", "presence_cities"]
cr_by_geo


,count,cr
other_regions,757924,0.025823
presence_cities,1102118,0.027894


In [57]:
# z-test: presence vs other
p = df[df["is_presence_city"] == 1]["target"]
o = df[df["is_presence_city"] == 0]["target"]

z, pval, p1, p2 = two_proportions_ztest(p.sum(), len(p), o.sum(), len(o))
print(f"Presence cities CR={p1:.5f}, Other regions CR={p2:.5f}")
print(f"Z={z:.3f}, p-value={pval:.5g}")


Presence cities CR=0.02789, Other regions CR=0.02582
Z=8.552, p-value=0


/var/folders/0q/s19p92dx2lx90d05rffjvmyw0000gn/T/ipykernel_15876/2329886848.py:9: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (Deprecated Numpy 1.25). Replace usages of `np.math` with `math`
  p_value = 2 * (1 - 0.5 * (1 + np.math.erf(abs(z)/np.sqrt(2))))


✅ Часть 2. Ответы продуктовой команде
1) Самый “целевой” трафик: источники/кампании/устройства/локации
Ячейка 10 — топ источников по объёму и CR

In [58]:
# Топ utm_source: объем + CR
traffic_source = (
    df.groupby("utm_source")["target"]
    .agg(visits="count", conversions="sum", cr="mean")
    .sort_values(["visits"], ascending=False)
)

traffic_source.head(20)


,visits,conversions,cr
utm_source,,,
ZpYIoDJMcFzVoPFsHGJL,578290,15998,0.027664
fDLlAcSmythWSCVMvqvL,300575,10531,0.035036
kjsLglQLzykiRbcDiGcD,266354,6293,0.023626
MvfHsxITijuriZxsqZqt,186199,2249,0.012078
BHcvLfOaCWvWTykYqHVe,116320,3882,0.033373
bByPQxmDaMXgpHeypKSM,102287,5557,0.054328
QxAxdyPLuQMEcrdZWdWb,51415,1404,0.027307
aXQzDWsJuGXeBXexNHjc,31152,1827,0.058648
jaSOmLICuBzCFqHfBdRg,29241,401,0.013714


Ячейка 11 — топ кампаний по CR (с фильтром по минимальному объёму)

In [59]:
# Топ utm_campaign по CR, но чтобы не ловить шум — поставим минимальный объем
MIN_VISITS = 5000

traffic_campaign = (
    df.groupby("utm_campaign")["target"]
    .agg(visits="count", conversions="sum", cr="mean")
    .query("visits >= @MIN_VISITS")
    .sort_values(["cr", "visits"], ascending=[False, False])
)

traffic_campaign.head(20)


,visits,conversions,cr
utm_campaign,,,
LTuZkdKfxRGVceoWkVyg,463481,19006,0.041007
gecBYcKZCPMcVYdSSzKP,134042,4545,0.033907
eimRuUrNhZLAYcwRrNXu,7784,258,0.033145
LEoPHuyFvzoNfnzGgfcd,324044,9348,0.028848
sbJRYgVfvcnqKJNDDYIr,20188,575,0.028482
jqlUOdZBNZYfInQVcZlS,6388,178,0.027865
WiILFRDQbcHDHNvHzGpX,5793,150,0.025893
MXqmDyetMTICSSitTjWV,7038,157,0.022307
zxoiLxhuSIFrCeTLQVWZ,12370,270,0.021827


Ячейка 12 — устройства: volume + CR

In [60]:
# Устройства: объем + CR
traffic_device = (
    df.groupby("device_category")["target"]
    .agg(visits="count", conversions="sum", cr="mean")
    .sort_values(["visits"], ascending=False)
)

traffic_device


,visits,conversions,cr
device_category,,,
mobile,1474871,38379,0.026022
desktop,366863,11513,0.031382
tablet,18308,422,0.023050


Ячейка 13 — города: volume + CR (фильтр по объёму)

In [61]:
# Города: объем + CR (ограничим по минимуму, чтобы не было "случайных" городов)
MIN_CITY_VISITS = 3000

traffic_city = (
    df.groupby("geo_city")["target"]
    .agg(visits="count", conversions="sum", cr="mean")
    .query("visits >= @MIN_CITY_VISITS")
    .sort_values(["cr", "visits"], ascending=[False, False])
)

traffic_city.head(20)


,visits,conversions,cr
geo_city,,,
Domodedovo,4038,263,0.065131
Stavropol,4377,180,0.041124
Kazan,29531,1139,0.038570
Korolyov,4047,145,0.035829
Sochi,8972,316,0.035221
Krasnodar,32243,1081,0.033527
Pyatigorsk,3782,124,0.032787
Khimki,7385,236,0.031957
Grozny,12742,401,0.031471


2) Какие авто пользуются спросом? (через события в hits)

У нас в sessions нет явной информации об авто, но она часто есть в hit_page_path (например, /cars/all/volkswagen/polo/...).

Ячейка 14 — вытаскиваем “марку/модель” из hit_page_path и считаем спрос/CR

In [62]:
# Забираем только те строки hits, где есть page_path с авто
# Дальше попытаемся вытащить brand/model из URL-пути вида:
# sberauto.com/cars/all/<brand>/<model>/...

hits_df = pd.read_sql("""
SELECT session_id, hit_page_path, event_action, hit_type
FROM hits
WHERE hit_page_path IS NOT NULL
LIMIT 500000; -- чтобы не убить память в ноутбуке (можно поднять/убрать, если тянет)
""", con)

hits_df.head()


,session_id,hit_page_path,event_action,hit_type
0,5639623078712724064.1640254056.1640254056,sberauto.com/cars?utm_source_initial=google&ut...,quiz_show,event
1,7750352294969115059.1640271109.1640271109,sberauto.com/cars/fiat?city=1&city=18&rental_c...,quiz_show,event
2,885342191847998240.1640235807.1640235807,sberauto.com/cars/all/volkswagen/polo/e994838f...,quiz_show,event
3,142526202120934167.1640211014.1640211014,sberauto.com/cars?utm_source_initial=yandex&ut...,quiz_show,event
4,3450086108837475701.1640265078.1640265078,sberauto.com/cars/all/mercedes-benz/cla-klasse...,quiz_show,event


In [63]:
# Простейший парсинг brand/model:
# Ищем кусок "/cars/all/<brand>/<model>/" (это эвристика)
import re

def extract_brand_model(url):
    if not isinstance(url, str):
        return (None, None)
    m = re.search(r"/cars/(?:all/)?([^/?]+)/([^/?]+)/", url)
    if m:
        return m.group(1), m.group(2)
    return (None, None)

hits_df[["brand", "model"]] = hits_df["hit_page_path"].apply(lambda x: pd.Series(extract_brand_model(x)))

hits_df[["brand", "model"]].head(10)


,brand,model
0,None,None
1,None,None
2,volkswagen,polo
3,None,None
4,mercedes-benz,cla-klasse
5,None,None
6,None,None
7,mercedes-benz,glc
8,kia,sorento
9,nissan,x-trail


In [64]:
# Теперь присоединим target по session_id из нашей витрины df
target_map = df.set_index("session_id")["target"]

hits_df["target"] = hits_df["session_id"].map(target_map).fillna(0).astype(int)

# "Спрос" = сколько раз встречается карточка авто (hits), можно считать как просмотры.
# А CR по авто — доля целевых сессий среди сессий, где этот авто встречался.
auto_stats = (
    hits_df.dropna(subset=["brand", "model"])
    .groupby(["brand", "model"])["target"]
    .agg(hit_cnt="count", uniq_sessions="size", cr="mean")
    .sort_values(["hit_cnt"], ascending=False)
)

auto_stats.head(20)


,,hit_cnt,uniq_sessions,cr
brand,model,,,
skoda,rapid,25919,25919,0.106100
volkswagen,polo,17921,17921,0.117683
lada-vaz,vesta,16949,16949,0.103369
skoda,karoq,12970,12970,0.069931
mercedes-benz,e-klasse,9560,9560,0.065377
nissan,qashqai,7704,7704,0.054517
kia,rio,7005,7005,0.129336
renault,duster,6961,6961,0.056601
nissan,x-trail,6923,6923,0.061390


3) Стоит ли увеличивать присутствие в соцсетях?

Мы делаем это как бизнес-вывод на основе utm_source/utm_medium, где соцсети обычно имеют узнаваемые названия (vk, facebook, instagram, tiktok и т.д.).

Ячейка 15 — выделяем соцсети и сравниваем их CR с остальными

In [65]:
# Составим список "социальных" источников по подстрокам
social_patterns = ["vk", "vkontakte", "facebook", "instagram", "tiktok", "ok", "odnoklassniki", "mytarget", "telegram"]

src = df["utm_source"].fillna("").str.lower()
df["is_social"] = src.apply(lambda s: any(p in s for p in social_patterns)).astype(int)

df["is_social"].value_counts()


is_social
0    1859084
1        958
Name: count, dtype: int64

In [66]:
# CR соцсети vs не соцсети
social_cr = df.groupby("is_social")["target"].agg(["count", "mean"])
social_cr.rename(columns={"mean": "cr"}, inplace=True)
social_cr.index = ["not_social", "social"]
social_cr


,count,cr
not_social,1859084,0.027042
social,958,0.041754


In [67]:
# z-test для долей: social vs not_social
soc = df[df["is_social"] == 1]["target"]
ns  = df[df["is_social"] == 0]["target"]

z, pval, p1, p2 = two_proportions_ztest(soc.sum(), len(soc), ns.sum(), len(ns))
print(f"Social CR={p1:.5f}, Not social CR={p2:.5f}")
print(f"Z={z:.3f}, p-value={pval:.5g}")


Social CR=0.04175, Not social CR=0.02704
Z=2.806, p-value=0.0050155


/var/folders/0q/s19p92dx2lx90d05rffjvmyw0000gn/T/ipykernel_15876/2329886848.py:9: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (Deprecated Numpy 1.25). Replace usages of `np.math` with `math`
  p_value = 2 * (1 - 0.5 * (1 + np.math.erf(abs(z)/np.sqrt(2))))


✅ Часть 3. Артефакты и итоговые критерии (что сохраняем)
Ячейка 16 — сохраняем ключевые таблицы в reports/ как CSV

In [68]:
# Сохраняем артефакты для сдачи в папку reports/
import os
os.makedirs("../reports", exist_ok=True)

cr_by_traffic.to_csv("../reports/da_cr_traffic_type.csv")
cr_by_device.to_csv("../reports/da_cr_device.csv")
cr_by_geo.to_csv("../reports/da_cr_geo_presence.csv")

traffic_source.head(50).to_csv("../reports/da_top_utm_source.csv")
traffic_campaign.head(50).to_csv("../reports/da_top_utm_campaign.csv")
traffic_device.to_csv("../reports/da_device_stats.csv")
traffic_city.head(50).to_csv("../reports/da_top_cities.csv")

# Если auto_stats посчитался адекватно — тоже сохраним
auto_stats.head(100).to_csv("../reports/da_auto_stats_top.csv")

print("Saved DA artifacts to ../reports/")


Saved DA artifacts to ../reports/
